# tune-off Colab runner
A thin launcher: no experiment logic lives here. After a disconnect, re-run all cells; finished trials are skipped.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# For a private repo, store a token in Colab Secrets as GITHUB_TOKEN.
from google.colab import userdata
REF = 'main'  # use a release tag (e.g. v1.0.0) for real runs
REPO = 'github.com/<org>/tune-off.git'
token = userdata.get('GITHUB_TOKEN')
!git clone --branch {REF} https://{token}@{REPO} /content/tune-off 2>/dev/null || (cd /content/tune-off && git fetch && git checkout {REF} && git pull)
%cd /content/tune-off

In [ ]:
# 1) the package (keeps Colab's PyTorch)
!pip install -q -e '.[tasks]'
# 2) Mamba kernels, compiled against that PyTorch (can take 10+ minutes the first time)
!pip install -q packaging ninja wheel
!pip install -q --no-build-isolation -r requirements/colab.txt

In [ ]:
# Stops here if the fast Mamba kernels do not work: never run the grid without them
!python -m mvt info
from mvt.core.env import mamba_ssm_version, mamba_ssm_import_error
assert mamba_ssm_version(), f'mamba_ssm does not import: {mamba_ssm_import_error()}'

In [ ]:
# First session only: GPU checks, then save the exact package versions
!pytest -m gpu -q
!pip freeze > /content/drive/MyDrive/colab-freeze.txt
# Commit that file as requirements/colab-freeze.txt through a pull request.

In [ ]:
import os
os.environ['MVT_OWNER'] = '<your-name>'
os.environ['MVT_RESULTS_ROOT'] = '/content/drive/MyDrive/mvt_runs'
os.environ['MVT_DATA_CACHE'] = '/content/mvt_cache'   # local disk, not Drive

In [ ]:
# Your assigned cells
!python -m mvt run --task sst2 --fraction 0.25 --scale 2m

In [ ]:
!python -m mvt status
!python -m mvt export --task sst2
# Commit results/*.csv through a pull request from a local clone.